In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 50)

In [2]:
cols = [
    "id", "loan_amnt", "term", "int_rate", "installment", "grade",
    "sub_grade", "emp_length", "home_ownership", "annual_inc",
    "verification_status", "issue_d", "loan_status", "purpose",
    "addr_state", "dti", "delinq_2yrs", "fico_range_low",
    "fico_range_high", "open_acc", "revol_util", "total_acc",
    "total_pymnt", "recoveries"
]

df = pd.read_csv("../data/accepted_2007_to_2018Q4.csv",
                 usecols=cols, low_memory=False)

## Check the size and first rows

In [3]:
df.shape

(2260701, 24)

In [4]:
df.head()

,id,loan_amnt,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,purpose,addr_state,dti,delinq_2yrs,fico_range_low,fico_range_high,open_acc,revol_util,total_acc,total_pymnt,recoveries
0,68407277,3600.0,36 months,13.99,123.03,C,C4,10+ years,MORTGAGE,55000.0,Not Verified,Dec-2015,Fully Paid,debt_consolidation,PA,5.91,0.0,675.0,679.0,7.0,29.7,13.0,4421.723917,0.0
1,68355089,24700.0,36 months,11.99,820.28,C,C1,10+ years,MORTGAGE,65000.0,Not Verified,Dec-2015,Fully Paid,small_business,SD,16.06,1.0,715.0,719.0,22.0,19.2,38.0,25679.660000,0.0
2,68341763,20000.0,60 months,10.78,432.66,B,B4,10+ years,MORTGAGE,63000.0,Not Verified,Dec-2015,Fully Paid,home_improvement,IL,10.78,0.0,695.0,699.0,6.0,56.2,18.0,22705.924294,0.0
3,66310712,35000.0,60 months,14.85,829.90,C,C5,10+ years,MORTGAGE,110000.0,Source Verified,Dec-2015,Current,debt_consolidation,NJ,17.06,0.0,785.0,789.0,13.0,11.6,17.0,31464.010000,0.0
4,68476807,10400.0,60 months,22.45,289.91,F,F1,3 years,MORTGAGE,104433.0,Source Verified,Dec-2015,Fully Paid,major_purchase,PA,25.37,1.0,695.0,699.0,12.0,64.5,35.0,11740.500000,0.0


## Check the column types

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2260701 entries, 0 to 2260700
Data columns (total 24 columns):
 #   Column               Dtype  
---  ------               -----  
 0   id                   object 
 1   loan_amnt            float64
 2   term                 object 
 3   int_rate             float64
 4   installment          float64
 5   grade                object 
 6   sub_grade            object 
 7   emp_length           object 
 8   home_ownership       object 
 9   annual_inc           float64
 10  verification_status  object 
 11  issue_d              object 
 12  loan_status          object 
 13  purpose              object 
 14  addr_state           object 
 15  dti                  float64
 16  delinq_2yrs          float64
 17  fico_range_low       float64
 18  fico_range_high      float64
 19  open_acc             float64
 20  revol_util           float64
 21  total_acc            float64
 22  total_pymnt          float64
 23  recoveries           float64
dty

## Check missing values

In [6]:
(df.isnull().mean() * 100).sort_values(ascending=False).head(10)

emp_length     6.499754
revol_util     0.081170
dti            0.077144
total_acc      0.002743
open_acc       0.002743
delinq_2yrs    0.002743
annual_inc     0.001637
loan_status    0.001460
purpose        0.001460
total_pymnt    0.001460
dtype: float64

## Check important column: loan status

I am checking this column because my project is about default risk, and loan_status tells me which loans were paid and which went bad. I need to see all its values before I decide what counts as a default.

In [7]:
df["loan_status"].value_counts()

loan_status
Fully Paid                                             1076751
Current                                                 878317
Charged Off                                             268559
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     40
Name: count, dtype: int64

## Make a copy

In [8]:
df_clean = df.copy()
print("Rows before cleaning:", len(df_clean))

Rows before cleaning: 2260701


## 3.2 Keep only loans with a final result

I am keeping only Fully Paid, Charged Off and Default loans. Current and Late loans do not have a final outcome yet, so they would give a wrong default rate. I am also leaving out about 2,700 loans from an old credit policy.

In [9]:
final_status = ["Fully Paid","Charged Off","Default"]
df_clean = df_clean[df_clean["loan_status"].isin(final_status)].copy()
print("Rows after keeping final loan:",len(df_clean))

Rows after keeping final loan: 1345350


## Create the default flag column

In [10]:
df_clean["default_flag"]= df_clean["loan_status"].isin(["Charged Off", "Default"]).astype(int)
df_clean["default_flag"].value_counts()

default_flag
0    1076751
1     268599
Name: count, dtype: int64

In [11]:
print("Default rate: {:.2f}%".format(df_clean["default_flag"].mean()*100))

Default rate: 19.96%


## Fix the date column

In [12]:
df_clean["issue_d"]= pd.to_datetime(df_clean["issue_d"],format="%b-%Y")

In [14]:
df_clean["issue_year"]= df_clean["issue_d"].dt.year
df_clean["issue_year"].value_counts().sort_index()

issue_year
2007       251
2008      1562
2009      4716
2010     11536
2011     21721
2012     53367
2013    134804
2014    223103
2015    375546
2016    293105
2017    169321
2018     56318
Name: count, dtype: int64

## Clean the term Column

In [17]:
df_clean["term_months"]= df_clean["term"].str.extract(r"(\d+)")[0].astype(int)
df_clean["term_months"].value_counts()

term_months
36    1020768
60     324582
Name: count, dtype: int64

## Clean employment length

In [19]:
emp_map = {"< 1 year": 0, "1 year": 1, "2 year": 2, "3 year": 3, "4 year": 4, "5 year": 5, "6 year": 6, 
           "7 year": 7, "8 year": 8, "9 year": 9, "10+ years": 10}

In [20]:
df_clean["emp_length_yrs"]= df_clean["emp_length"].map(emp_map)

In [21]:
df_clean["emp_length"]= df_clean["emp_length"].fillna("Unknown")

## 3.7 Check for strange values

I am checking the minimum, maximum and average of the main columns. I want to find errors before I remove any rows.

In [26]:
df_clean[["loan_amnt", "annual_inc", "dti", "int_rate", "revol_util"]].describe().round(2).T

,count,mean,std,min,25%,50%,75%,max
loan_amnt,1345350.0,14419.97,8717.10,500.00,8000.00,12000.00,20000.00,40000.00
annual_inc,1345350.0,76247.57,69924.85,0.00,45780.00,65000.00,90000.00,10999200.00
dti,1344976.0,18.28,11.16,-1.00,11.79,17.61,24.06,999.00
int_rate,1345350.0,13.24,4.77,5.31,9.75,12.74,15.99,30.99
revol_util,1344493.0,51.81,24.52,0.00,33.40,52.20,70.70,892.30


## Handle missing values and errors

I removed rows with missing key values and rows with impossible values (zero income, negative or very high DTI). I printed how many rows each step removed.
I removed rows with blank key values, zero income, and impossible DTI values (below 0 or above 100). I printed how many rows each rule removed.

In [27]:
print("Income zero or less:", (df_clean["annual_inc"] <= 0).sum())
print("DTI below 0:", (df_clean["dti"] < 0).sum())
print("DTI above 100:", (df_clean["dti"] > 100).sum())
print("DTI missing:", df_clean["dti"].isnull().sum())
print("revol_util missing:", df_clean["revol_util"].isnull().sum())

Income zero or less: 361
DTI below 0: 2
DTI above 100: 533
DTI missing: 374
revol_util missing: 857


In [28]:
cols_needed = ["dti", "revol_util", "annual_inc", "delinq_2yrs", "open_acc", "total_acc"]

rows_start = len(df_clean)

df_clean = df_clean.dropna(subset=cols_needed)
rows_after_missing = len(df_clean)

df_clean = df_clean[(df_clean["annual_inc"] > 0) &
                    (df_clean["dti"] >= 0) &
                    (df_clean["dti"] <= 100)]
rows_end = len(df_clean)

print("Start:", rows_start)
print("Removed for missing values:", rows_start - rows_after_missing)
print("Removed for errors:", rows_after_missing - rows_end)
print("Final rows:", rows_end)

Start: 1345350
Removed for missing values: 1230
Removed for errors: 535
Final rows: 1343585


## Check for duplicates

In [29]:
df_clean["id"].duplicated().sum()

np.int64(0)

## Create average credit score column

In [30]:
df_clean["fico_avg"] = (df_clean["fico_range_low"] + df_clean["fico_range_high"]) / 2

## Create income groups columns

In [31]:
df_clean["income_bucket"] = pd.cut(
    df_clean["annual_inc"],
    bins=[0, 40000, 70000, 100000, 150000, float("inf")],
    labels=["<40K", "40-70K", "70-100K", "100-150K", "150K+"]
)

## Create DTI groups Column

In [32]:
df_clean["dti_bucket"] = pd.cut(
    df_clean["dti"],
    bins=[0, 10, 20, 30, 100],
    labels=["0-10", "10-20", "20-30", "30+"],
    include_lowest=True
)

In [33]:
df_clean["income_bucket"].value_counts().sort_index()
df_clean["dti_bucket"].value_counts().sort_index()
df_clean[["annual_inc", "income_bucket", "dti", "dti_bucket", "fico_avg"]].head(10)

,annual_inc,income_bucket,dti,dti_bucket,fico_avg
0,55000.0,40-70K,5.91,0-10,677.0
1,65000.0,40-70K,16.06,10-20,717.0
2,63000.0,40-70K,10.78,10-20,697.0
4,104433.0,100-150K,25.37,20-30,697.0
5,34000.0,<40K,10.20,10-20,692.0
6,180000.0,150K+,14.67,10-20,682.0
7,85000.0,70-100K,17.61,10-20,707.0
8,85000.0,70-100K,13.07,10-20,687.0
9,42000.0,40-70K,34.80,30+,702.0
12,64000.0,40-70K,34.95,30+,702.0


## Final Check

In [34]:
df_clean.info()
df_clean.head()

<class 'pandas.core.frame.DataFrame'>
Index: 1343585 entries, 0 to 2260697
Data columns (total 31 columns):
 #   Column               Non-Null Count    Dtype         
---  ------               --------------    -----         
 0   id                   1343585 non-null  object        
 1   loan_amnt            1343585 non-null  float64       
 2   term                 1343585 non-null  object        
 3   int_rate             1343585 non-null  float64       
 4   installment          1343585 non-null  float64       
 5   grade                1343585 non-null  object        
 6   sub_grade            1343585 non-null  object        
 7   emp_length           1343585 non-null  object        
 8   home_ownership       1343585 non-null  object        
 9   annual_inc           1343585 non-null  float64       
 10  verification_status  1343585 non-null  object        
 11  issue_d              1343585 non-null  datetime64[ns]
 12  loan_status          1343585 non-null  object        
 13  pu

,id,loan_amnt,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,purpose,addr_state,dti,delinq_2yrs,fico_range_low,fico_range_high,open_acc,revol_util,total_acc,total_pymnt,recoveries,default_flag,issue_year,term_months,emp_length_yrs,fico_avg,income_bucket,dti_bucket
0,68407277,3600.0,36 months,13.99,123.03,C,C4,10+ years,MORTGAGE,55000.0,Not Verified,2015-12-01,Fully Paid,debt_consolidation,PA,5.91,0.0,675.0,679.0,7.0,29.7,13.0,4421.723917,0.0,0,2015,36,10.0,677.0,40-70K,0-10
1,68355089,24700.0,36 months,11.99,820.28,C,C1,10+ years,MORTGAGE,65000.0,Not Verified,2015-12-01,Fully Paid,small_business,SD,16.06,1.0,715.0,719.0,22.0,19.2,38.0,25679.660000,0.0,0,2015,36,10.0,717.0,40-70K,10-20
2,68341763,20000.0,60 months,10.78,432.66,B,B4,10+ years,MORTGAGE,63000.0,Not Verified,2015-12-01,Fully Paid,home_improvement,IL,10.78,0.0,695.0,699.0,6.0,56.2,18.0,22705.924294,0.0,0,2015,60,10.0,697.0,40-70K,10-20
4,68476807,10400.0,60 months,22.45,289.91,F,F1,3 years,MORTGAGE,104433.0,Source Verified,2015-12-01,Fully Paid,major_purchase,PA,25.37,1.0,695.0,699.0,12.0,64.5,35.0,11740.500000,0.0,0,2015,60,NaN,697.0,100-150K,20-30
5,68426831,11950.0,36 months,13.44,405.18,C,C3,4 years,RENT,34000.0,Source Verified,2015-12-01,Fully Paid,debt_consolidation,GA,10.20,0.0,690.0,694.0,5.0,68.4,6.0,13708.948530,0.0,0,2015,36,NaN,692.0,<40K,10-20


## Saving this file

In [35]:
import os
os.makedirs("../data/sample", exist_ok=True)

In [36]:
df_clean.to_csv("../data/loan_clean.csv", index=False)

In [37]:
df_clean.sample(10000, random_state=42).to_csv("../data/sample/loan_sample.csv", index=False)